# E-Commerce Lakehouse: Silver Order Items

## Convert source data types

Read the saved Bronze order items and convert:
- order_item_id to an integer.
- shipping_limit_date to a timestamp.
- price and freight_value to decimal values with two decimal places.

Preserve identifiers as strings and retain ingestion metadata.
Validate conversions before saving the Silver table.

In [ ]:
from pyspark.sql.functions import col, to_timestamp

bronze_order_items_df = spark.table(
    "workspace.ecommerce.bronze_order_items"
)

silver_order_items_df = (
    bronze_order_items_df
    .withColumn("order_item_id", col("order_item_id").cast("int"))
    .withColumn(
        "shipping_limit_date",
        to_timestamp("shipping_limit_date", "yyyy-MM-dd HH:mm:ss")
    )
    .withColumn("price", col("price").cast("decimal(12,2)"))
    .withColumn("freight_value", col("freight_value").cast("decimal(12,2)"))
)

silver_order_items_df.printSchema()
display(silver_order_items_df.limit(5))

## Validate Silver order items

Compare source and Silver null counts for converted columns.
Check required identifiers, duplicate order-item keys, and invalid
numeric values. Preserve records while investigating any issues.

In [ ]:
from pyspark.sql.functions import trim

print("Bronze rows:", bronze_order_items_df.count())
print("Silver rows:", silver_order_items_df.count())

# Check whether conversions introduced additional nulls.
for column_name in [
    "order_item_id",
    "shipping_limit_date",
    "price",
    "freight_value"
]:
    bronze_nulls = bronze_order_items_df.filter(
        col(column_name).isNull()
    ).count()

    silver_nulls = silver_order_items_df.filter(
        col(column_name).isNull()
    ).count()

    print(
        f"{column_name}: "
        f"Bronze nulls={bronze_nulls}, Silver nulls={silver_nulls}"
    )

# Check required text identifiers.
for column_name in ["order_id", "product_id", "seller_id"]:
    missing_count = silver_order_items_df.filter(
        col(column_name).isNull()
        | (trim(col(column_name)) == "")
    ).count()

    print(f"Missing or blank {column_name}: {missing_count}")

# An order can have multiple items, but each item key should be unique.
duplicate_item_keys_df = (
    silver_order_items_df
    .groupBy("order_id", "order_item_id")
    .count()
    .filter(col("count") > 1)
)

print("Duplicate order-item keys:", duplicate_item_keys_df.count())

# Zero freight is allowed; negative amounts are flagged.
invalid_values_df = silver_order_items_df.filter(
    (col("order_item_id") <= 0)
    | (col("price") < 0)
    | (col("freight_value") < 0)
)

print("Rows with invalid numeric values:", invalid_values_df.count())

## Validate order relationships and save Silver order items

Check that every order item references an existing Silver order.
Save the validated data as a managed Delta table and verify
the stored row count. Leave an existing table unchanged.

In [ ]:
silver_orders_df = spark.table("workspace.ecommerce.silver_orders")

unmatched_count = silver_order_items_df.join(
    silver_orders_df.select("order_id"),
    on="order_id",
    how="left_anti"
).count()

row_count = silver_order_items_df.count()

duplicate_key_count = (
    silver_order_items_df
    .groupBy("order_id", "order_item_id")
    .count()
    .filter("count > 1")
    .count()
)

print("Items without a matching order:", unmatched_count)
print("Silver item rows:", row_count)
print("Duplicate order-item keys:", duplicate_key_count)

if (
    unmatched_count != 0
    or row_count != 112650
    or duplicate_key_count != 0
):
    raise ValueError(
        "Silver item validation failed. Stopped before replacing the table."
    )

table_name = "workspace.ecommerce.silver_order_items"

(
    silver_order_items_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

saved_silver_order_items_df = spark.table(table_name)
saved_count = saved_silver_order_items_df.count()

if saved_count != row_count:
    raise ValueError("Saved Silver item count does not match.")

print("Silver order items refreshed:", saved_count)